# 第2回 宿題 発展: 用語集を LoRA で学習させ、検索と比べる

講座の用語集（65 語）を、小さな指示チューニング済みモデルに LoRA で学習させます。同じテスト問題 22 問を、次の 3 つの方式で聞いて、正解の数を比べます。

| 方式 | 用語の知識の置き場所 | モデルに渡すもの |
| --- | --- | --- |
| 学習前 | どこにも無い | 質問だけ |
| 検索あり | プロンプト | 講座の `search_glossary` の検索結果と質問（学習前のモデル） |
| 学習後 | モデルの重み（adapter） | 質問だけ |

このノートブックは、演習のノートブック（`lora_format_tuning.ipynb`）のセクション 1〜5 と 7 を、発展用に書き換えたものです。セクション 0 は演習と同じです。演習のセクション 6（知識の質問）と 8（4bit 量子化）は、発展では使わないので入れていません。

**テスト問題** は、学習の効果を測るためにモデルに聞く質問の一覧です。学習データとは別に用意し、3 種類に分けます。

| 種類 | 問題の数 | 確かめること |
| --- | --- | --- |
| 同じ文 | 8 | 学習データとまったく同じ文で聞いて、覚えたか |
| 言い換え | 8（別名で聞く 4・用語名も別名も出さずに聞く 4） | 初めて見る文で聞いて、使える知識になったか |
| 知らない語 | 6 | 用語集に無い語を聞いて、知らないと言えるか |

**採点** は、答えの文字列に決めた語句が入っているかで ○× を付けることです。同じ文と言い換えは `must_include` の語句をすべて含めば ○、知らない語は `must_any` の語句（「分かりません」など）をどれか 1 つ含めば ○ にします。

**始める前に**: 手元で作った次の 2 つのファイルを、左のファイル一覧（フォルダのアイコン）から `/content` にアップロードします。

- `glossary_sft.jsonl`: 学習データ 130 件（`scripts/make_sft_dataset.py` で作る）
- `glossary_test.json`: テスト問題 22 問と検索結果（`lectures/02/advanced/make_glossary_test.py` で作る）

**進め方**

| セクション | 内容 | 所要時間の目安（T4） |
| --- | --- | --- |
| 0〜2 | 環境、設定、データの読み込み | 約 3 分 |
| 3 | モデルの読み込みと、学習前・検索ありの評価 | 約 10 分（推測） |
| 4 | LoRA 学習（5 エポック、約 85 ステップ） | 約 9 分（推測） |
| 5・7 | 学習後の評価と、`result.json` の書き出し | 約 5 分（推測） |

セルは上から順に実行してください。途中のセルを飛ばすと、後のセルで変数が見つからないエラーになります。

## 0. 環境の確認とライブラリの導入

**やること**: GPU が割り当てられているかを確認し、ライブラリを動作確認済みの版に固定して導入します。

**見るところ**

- 1 つ目のセルに `Tesla T4` と表示されること。
- 2 つ目のセルの最後に、版の一覧（`transformers: 5.18.0` など）が表示されること。導入は 1 分ほどかかります。

**うまくいかないとき**

- 「GPU が見つかりません」と出る: メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選び、セルを実行し直します。混雑で割り当てられないときは、時間をおいて試します。
- pip の赤い `ERROR: pip's dependency resolver ...`（`cudf` や `pyarrow` に関するもの）は、Colab に最初から入っている別のパッケージとの食い違いの警告です。この演習には影響しないので、そのまま進めます。

In [1]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

name, memory.total [MiB]
Tesla T4, 15360 MiB



In [2]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
{'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0', 'torch': '2.14.1+cu130'}


## 1. 設定

**やること**: 演習と同じ `CONFIG` を使います。変えたのは次の 2 つだけです。

- `num_train_epochs: 5`: 学習データ 130 件を 5 周します。1 周が約 17 ステップなので、合計約 85 ステップです。
- `max_new_tokens: 256`: 用語の定義文は最長 212 文字あるので、答えが途中で切れないように増やしました。

**エポック数を 5 から始める理由**

演習の 20 のままにしないのは、学習データの件数が 14 件から 130 件に増えたためです。

| 比べるもの | 演習（20 エポック） | 発展を 20 にした場合 | 発展（5 エポック） |
| --- | --- | --- | --- |
| 学習データ | 14 件 | 130 件 | 130 件 |
| ステップ数（1 ステップ 8 件） | 40 | 約 340 | 約 85 |
| 学習時間（1 ステップ約 6 秒で見積もり） | 約 4 分（実測 240.5 秒） | 約 34 分 | 約 9 分 |
| 1 つの語を学習する回数 | 20 回（1 件を 20 周） | 40 回（ひな形 2 つ × 20 周） | 10 回（ひな形 2 つ × 5 周） |

- 20 のままだと、学習だけで 30 分を超え、Colab の無料枠では時間の余裕がなくなります。
- 5 なら、ステップ数は演習の 2 倍ほどで、学習は 10 分ほどに収まる見込みです。
- 1 つの語を学習する回数は 10 回で、演習の 1 件あたり 20 回の半分です。足りるかは分からないので、5 は「まず試す値」です。
- 学習しすぎると、学習データに無い質問にも、覚えた定義文に似た文で答えやすくなる（知らない語で作り話をする）傾向が一般に知られています（推測。この宿題では確かめていません）。そのため、少ない回数から始めて、足りなければ増やします。

**足りなかったときの基準**: セクション 5 で、同じ文の質問（学習データとまったく同じ文）の正解が 8 問中 4 問未満なら、10 にして学習し直します。学習した文そのままにすら答えられないなら、言い換えや検索と比べる前に、学習が足りていないと判断できるためです。

**見るところ**: 設定の JSON が表示されます。

In [3]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # 小さな指示チューニング済みモデル（Apache-2.0）。量子化なしで T4 に載る大きさ
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # いつ実行しても同じ重みを使うために固定
    "seed": 42,
    "num_train_epochs": 10,   # 学習データ（130 件）を何周学習するか。1 周約 17 ステップ。セクション 5 の基準で 10 に上げることがある
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA の行列の次元（小さいほど学習するパラメータが少ない）
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 256,   # 定義文（最長 212 文字）が途中で切れないように増やす
    "gen": {"do_sample": False},   # 評価は貪欲生成で固定（前後比較の条件を揃える）
    "work_dir": "/content/qlora_lab",                          # 結果ファイルの保存先（左のファイル一覧から見られる）
    "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab",  # USE_DRIVE=True のときの保存先
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # True にすると Google Drive をマウントして adapter と出力を保存する
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

{
  "base_model": "Qwen/Qwen3.5-0.8B",
  "revision": "2fc06364715b967f1860aea9cf38778875588b17",
  "seed": 42,
  "num_train_epochs": 10,
  "warmup_steps": 6,
  "lr": 0.0002,
  "lora_r": 16,
  "lora_alpha": 32,
  "max_seq_len": 512,
  "max_new_tokens": 256,
  "gen": {
    "do_sample": false
  },
  "work_dir": "/content/qlora_lab",
  "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab"
}


## 2. 学習データとテスト問題の読み込み

**やること**: アップロードした 2 つのファイルを読み込みます。演習のように FAQ からデータを作ることはしません。

- 学習データは 1 行 1 件で、`{"messages": [user, assistant]}` の形です。user が質問、assistant が用語の定義文です。system はありません。
- テスト問題は 22 問で、1 問ごとに質問の文・種類・採点の語句・検索結果を持ちます。

読み込んだあとに、件数と次の 2 つを確かめます。違うファイルをアップロードしていたら、ここで止まります。

- 同じ文の質問は、学習データにあること。
- 言い換えと知らない語の質問は、学習データに無いこと。

**見るところ**: `学習データ 130 件 | テスト問題 22 問` と、学習データの 1 件目が表示されます。

**うまくいかないとき**: `FileNotFoundError` が出たら、2 つのファイルが `/content` に（フォルダに入れずに）アップロードされているかを確かめます。

### このセルでしていること

上から順に、次の 4 つをします。

1. **ファイルがあるかを確かめる**: `/content` に 2 つのファイルが無ければ、無いファイルの名前を出して止まります（`FileNotFoundError`）。
2. **読み込む**: 学習データを `train_rows`、テスト問題を `test_items` に入れます。
3. **件数を確かめる**: 学習データが 130 件、テスト問題が 22 問でなければ止まります（`assert`）。
4. **質問の重なりを確かめる**: 同じ文の質問は学習データにあり、言い換えと知らない語の質問は学習データに無いことを確かめます。

読み込んだデータの 1 件は、次の形です。

| 変数 | 1 件の例 |
| --- | --- |
| `train_rows` | `{"messages": [{"role": "user", "content": "「LoRA」とは何ですか？"}, {"role": "assistant", "content": "（LoRA の定義文）"}]}` |
| `test_items` | `{"kind": "same", "term_id": "lora", "question": "「LoRA」とは何ですか？", "must_include": ["行列", "固定"], "search_result": "（検索結果）"}` |

`train_questions` は、学習データ 130 件の質問（user の文）だけを集めた集合です。4. の確認で、テスト問題の質問がこの中にあるかを調べるのに使います。

**ログ（進み具合の表示）**: このセルで `log()` という関数を作ります。`[14:25:12] 読み込み完了: …` のように、日本時間の時刻を付けて 1 行表示する関数です。この後のセルでも、時間のかかる処理の始まりと終わりにこの関数を使います。時刻が付くので、表示が止まって見えても、最後のログからどれだけたったかで、止まっているかを判断できます。

このセルのログ（例）:

```text
[14:25:12] 読み込み完了: 学習データ 130 件 | テスト問題 22 問
```

**`print` ではなく `log` を使う理由**

`log()` の中身は `print()` です。`print()` をそのまま使わずに `log()` を通すのは、次の 3 つのためです。

| 理由 | `print()` だけの場合 | `log()` の場合 |
| --- | --- | --- |
| 時刻が付く | 何時に出た行か分からない | `[14:27:24]` が付くので、最後の行からどれだけたったかで、止まっているかを判断できる |
| すぐに画面に出る | 出力がためられて、長い処理の途中では遅れて出ることがある | `flush=True` で、呼んだその時に画面に出す |
| 形を 1 か所で決められる | 時刻の付け方を、呼ぶ所ごとに書く必要がある | 形を変えたいときは、`log()` の 1 か所を直せばよい |

- 進み具合を伝える行（始まり・終わり・1 問ごと）だけを `log()` にしています。学習データの 1 件目や `result.json` の中身のような「結果そのもの」は、`print()` のままです。時刻の付いた行を追えば、進み具合だけを読めます。
- Python には標準の `logging` モジュールもありますが、使っていません。ノートブックでは、出力先や表示の形の設定を先に書く必要があり、ここで欲しいのは「時刻付きで、すぐ出る 1 行」だけだからです。`log()` は、そのための小さな関数です。

In [5]:
from datetime import datetime
from zoneinfo import ZoneInfo


def log(message: str) -> None:
    """
    進み具合のログを、日本時間の時刻つきで 1 行表示する関数。

    flush=True で、長い処理の途中でもすぐに画面に出す。
    """
    print(f"[{datetime.now(ZoneInfo('Asia/Tokyo')):%H:%M:%S}] {message}", flush=True)


# 手元で作った 2 つのファイルの置き場所（左のファイル一覧から /content にアップロードしておく）
TRAIN_FILE = Path("/content/glossary_sft.jsonl")
TEST_FILE = Path("/content/glossary_test.json")
# missing（見つからないファイルのリスト）: 2 つのうち、/content に無いファイルの名前
missing = [path.name for path in (TRAIN_FILE, TEST_FILE) if not path.exists()]
# アップロードしていないファイルがある場合: 何をすればよいかを書いたエラーで止める
if missing:
    raise FileNotFoundError(f"{', '.join(missing)} がありません。左のファイル一覧から /content にアップロードしてから、このセルを実行し直してください")
# train_rows（学習データ）: 1 行 1 件の {"messages": [user, assistant]} が 130 件
train_rows = [json.loads(line) for line in TRAIN_FILE.read_text(encoding="utf-8").splitlines() if line.strip()]
# test_items（テスト問題）: kind・question・must_include か must_any・search_result を持つ辞書が 22 問
test_items = json.loads(TEST_FILE.read_text(encoding="utf-8"))
# 件数が違う場合は、別のファイルをアップロードしているので止める
assert len(train_rows) == 130, f"学習データが 130 件ではない: {len(train_rows)}"
assert len(test_items) == 22, f"テスト問題が 22 問ではない: {len(test_items)}"

# train_questions（学習データの質問）: 学習データ 130 件の user の文の集合
train_questions = {row["messages"][0]["content"] for row in train_rows}
# 約束 1・2 を Colab でも確かめる（アップロードしたファイルの取り違えに気づくため）
for item in test_items:
    in_train = item["question"] in train_questions
    # 同じ文の質問の場合: 学習データにあるはず
    if item["kind"] == "same":
        assert in_train, f"同じ文の質問が学習データに無い: {item['question']}"
    # 言い換え・知らない語の質問の場合: 学習データに無いはず
    else:
        assert not in_train, f"学習データにある質問がテストに入っている: {item['question']}"

# 読み込みの終わりと件数を表示する
log(f"読み込み完了: 学習データ {len(train_rows)} 件 | テスト問題 {len(test_items)} 問")
# 学習データの 1 件目を表示する
print("学習データの 1 件:", train_rows[0]["messages"])

[15:38:41] 読み込み完了: 学習データ 130 件 | テスト問題 22 問
学習データの 1 件: [{'role': 'user', 'content': '「AIエンジニアリング」とは何ですか？'}, {'role': 'assistant', 'content': '学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを一から作ることより、モデルの選択、データやツールとの接続、評価と改善のサイクルを回すことに重心がある。'}]


## 3. モデルの読み込みと、学習前・検索ありの評価

**やること**: モデルを演習と同じく量子化なしで読み込み、22 問に 2 つの方式で答えさせて採点します。

- **学習前**: 質問だけを渡します。
- **検索あり**: 講座の `search_glossary` の検索結果を、質問の前に入れて渡します。

どちらの方式でも、答え方の指示（「簡潔に」「知らなければ分からないと答えて」など）は足しません。指示を足すと、検索の効果と指示の効果が混ざるためです。

検索ありを学習の前に済ませるのは、学習すると変数 `model` にも adapter が入るためです。

**見るところ**

- 2 つの方式の要約（`{"label": "before", "同じ文": "0/8", ...}` の形）。正解の数を「正解の数/問題の数」で表します。
- 1 問ごとの表。○× だけでなく、答えの原文も読みます。語句が入っているだけの見当違いの答えや、語句を言い換えた正しい答えがあるためです。

### このセルでしていること：モデルを読み込み、答えさせる関数を作る

| 部分 | 中身 |
| --- | --- |
| モデルの読み込み | 演習と同じ。トークナイザー（文字列とトークンを変換する部品）とモデルを読み込み、GPU に載せます |
| `build_messages` | モデルに渡すメッセージを作ります。2 つの版があります |
| `generate` | 1 問に答えさせ、答えの文字列を返します |

`build_messages` が作るメッセージは、方式によって次のように変わります。

| 方式 | `with_search` | user に入れる文 |
| --- | --- | --- |
| 学習前・学習後 | `False` | `「LoRA」とは何ですか？`（質問だけ） |
| 検索あり | `True` | `用語集の検索結果:\n[{"term": "LoRA", ...}]\n\n質問: 「LoRA」とは何ですか？` |

- 演習の `SYSTEM`（JSON で答える指示）は消しました。学習データに system が無いので、評価でも入れません。
- どちらの版にも、答え方の指示（「簡潔に」など）は足しません。足すと、検索の効果と指示の効果が混ざるためです。

このセルでは、まだ評価はしません。モデルの読み込みに 1 分ほどかかります。

**ログ**: 読み込みの始まりと終わりに 1 行ずつ出ます。初回はモデルのダウンロードがあるので、2 行の間に 1〜2 分かかります。

```text
[14:26:01] モデルの読み込みを始める: Qwen/Qwen3.5-0.8B
[14:27:10] モデルの読み込み完了 | device: cuda | 重みのメモリ(GB): 3.2
```

（時刻とメモリの値は例です。）

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 使える計算装置を選ぶ（Colab の T4 なら cuda）
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
# 読み込みの開始を表示する（初回はダウンロードがあるので 1〜2 分かかる）
log(f"モデルの読み込みを始める: {CONFIG['base_model']}")
# トークナイザー（文字列とトークンを変換する部品）を読み込む
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
# 埋め草のトークンが無い場合: 文の終わりのトークンで代用する
if tok.pad_token is None: tok.pad_token = tok.eos_token
# 量子化なしで、重みを 32bit のまま読み込み、GPU に載せる
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
# 生成を速くするキャッシュを有効にする
model.config.use_cache = True
# 重みが使うメモリを GB で記録する
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
# 読み込みの終わりと、使う装置・重みのメモリを表示する
log(f"モデルの読み込み完了 | device: {DEVICE} | 重みのメモリ(GB): {BASE_MEMORY_GB}")


def build_messages(item: dict, with_search: bool = False) -> list[dict]:
    """
    モデルに渡すメッセージを作る関数。

    学習データと同じく system は入れず、user に質問だけを入れる。
    答え方の指示は、どちらの版にも足さない（約束 3）。

    Args:
        item: テスト問題 1 問の辞書。
        with_search: True なら、講座の検索結果を質問の前に入れる（検索あり）。

    Returns:
        user の 1 件だけのメッセージのリスト。
    """
    # 検索ありの場合: 検索結果と質問を入れる
    if with_search:
        content = f"用語集の検索結果:\n{item['search_result']}\n\n質問: {item['question']}"
    # 学習前・学習後の場合: 質問だけを入れる
    else:
        content = item["question"]
    return [{"role": "user", "content": content}]


def generate(model, item: dict, with_search: bool = False) -> str:
    """
    1 問に答えさせ、答えの文字列を返す関数。

    Args:
        model: 答えさせるモデル（学習前の model か、学習後の tuned）。
        item: テスト問題 1 問の辞書。
        with_search: True なら、検索ありの版で質問する。

    Returns:
        モデルが生成した答えの文字列（前後の空白は除く）。
    """
    # メッセージをモデル用の文字列にする。enable_thinking=False で、思考ブロックを空にして答えだけを生成させる
    prompt = tok.apply_chat_template(build_messages(item, with_search), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    # 文字列をトークンに変え、モデルと同じ装置に載せる
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    # 学習はしないので、勾配を計算せずに生成する
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    # 入力の部分を除き、生成した部分だけを文字列に戻す
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

[15:38:46] モデルの読み込みを始める: Qwen/Qwen3.5-0.8B


config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

[15:39:18] モデルの読み込み完了 | device: cuda | 重みのメモリ(GB): 3.01


### このセルでしていること：22 問に答えさせて採点する（学習前・検索あり）

| 部分 | 中身 |
| --- | --- |
| 準備 | 表の中の答えを省略せずに出す設定と、2 つの定数 |
| `group_of` | 1 問を、結果の表の 4 つの種類（同じ文・言い換え（別名）・言い換え（用語名なし）・知らない語）に分けます |
| `score` | 答えに ○× を付けます（採点） |
| `evaluate` | 22 問に答えさせて採点し、種類ごとの正解の数を数えます |
| `mark` | `True` を ○、`False` を × の記号にします |
| 実行 | 学習前と検索ありで `evaluate` を 1 回ずつ呼び、2 つの答えを並べた表を出します |

**種類を 4 つに分ける理由**: テスト問題の `kind` は 3 種類ですが、言い換えの 8 問は「別名で聞いた 4 問」と「用語名も別名も出さずに聞いた 4 問」で、検索の成績が大きく違うと予想しています。そのため、`ALIAS_TERM_IDS`（別名で聞いた 4 問の語の id）を使って分けて数えます。

**`group_of` という名前の付け方**

- **group**（グループ）は「まとまり、組」という意味です。ここでは、結果の表で 1 行にまとめて数える単位（同じ文・言い換え（別名）・言い換え（用語名なし）・知らない語の 4 つ）を指します。
- **of** は「〜の」という意味です。`group_of(item)` と書くと、英語では「item の group」と読めます。
- 関数の動作は「テスト問題 1 問（item）を受け取り、その問題が入る group を返す」なので、呼び出す行を読むだけで、何が返るかが分かるように名前を付けました。例えば `group_of(item)` が `"alias"` を返したら、「この問題の group は alias（言い換え（別名））」と読めます。
- 動詞で始まる名前（`classify_item` など）にしなかったのは、この関数が何かを書き換えるのではなく、値を調べて返すだけだからです。
- `kind_of` にしなかったのは、テスト問題がもともと持つ `kind`（same・paraphrase・unknown の 3 種類）と区別するためです。`group` は、`kind` の paraphrase を 2 つに分けた、表のための 4 つの分け方です。同じ言葉を使うと、3 種類の話か 4 種類の話かが紛れます。
- 結果の辞書のキー `"group"` と、呼び名の対応 `GROUP_NAMES` も、同じ言葉にそろえています。

**採点（`score`）のしかた**

| 質問の種類 | ○ になる条件 | 例 |
| --- | --- | --- |
| 同じ文・言い換え | `must_include` の語句を**すべて**答えに含む（`all`） | LoRA の質問なら「行列」と「固定」の両方 |
| 知らない語 | `must_any` の語句を**どれか 1 つ**含む（`any`） | 「その用語は分かりません。」なら「分かりません」を含むので ○ |

**`evaluate` が返すもの**

- `summary`（方式ごとのまとめ）: 例えば `{"label": "before", "n": 22, "同じ文": "0/8", "言い換え（別名）": "0/4", "言い換え（用語名なし）": "0/4", "知らない語": "6/6", "合計": "6/22", "seconds": 312.4}`。正解の数は「正解の数/問題の数」で表します。
- `rows`（1 問ごとの結果）: 例えば `{"group": "same", "term_id": "lora", "question": "「LoRA」とは何ですか？", "raw": "（モデルの答え）", "ok": True}` が 22 件。
- 2 つとも `/content/qlora_lab/eval_before.json` のようなファイルにも保存します。

**学習の前に 2 つとも済ませる理由**: 学習すると、変数 `model` にも adapter（LoRA で足した部分）が入り、学習前のモデルが無くなるためです。

**見るところ**: 表の ○× だけでなく、答えの原文を読みます。語句を含むのに中身が違う答えや、語句を言い換えた正しい答えは、採点では見分けられないためです。

**ログ**: 1 問答えるごとに、番号・○×・経過秒数・質問を 1 行ずつ出します。22 問 × 2 方式で、合わせて 44 行と、方式ごとの始まりと終わりの行が出ます。

```text
[14:27:15] before の評価を始める（22 問、1 問ずつ進み具合を表示する）
[14:27:24] before: 1/22 問 | × | 経過 9 秒 | 「宿題の3レベル」とは何ですか？
[14:27:31] before: 2/22 問 | × | 経過 16 秒 | 「answer.md」とは何ですか？
…
[14:30:40] before の評価完了: {"label": "before", "同じ文": "0/8", …}
```

（時刻と秒数は例です。）

- 生成の前に、`causal_conv1d` や `flash-linear-attention` が入っていないという `[transformers]` の警告が 4 行出ます。高速化用のライブラリが無いので遅い代わりの実装を使う、という知らせで、計算結果は正しいままです。エラーではありません。
- 1 問に数秒〜20 秒ほどかかります（推測）。同じ番号のまま数分以上進まなければ、止まっているかもしれません。

In [7]:
import pandas as pd
# 表の中の答えの原文を、省略せずに表示する
pd.set_option("display.max_colwidth", None)

# 別名で聞く言い換えの質問の語。残りの言い換え 4 問は、用語名も別名も出さずに聞く質問
ALIAS_TERM_IDS = {"homework-levels", "default-model", "lora", "temperature"}
# 結果の表で使う、質問の種類の呼び名（言い換えは 2 つに分ける）
GROUP_NAMES = {"same": "同じ文", "alias": "言い換え（別名）", "no_name": "言い換え（用語名なし）", "unknown": "知らない語"}


def group_of(item: dict) -> str:
    """
    テスト問題 1 問を、結果の表の 4 つの種類に分ける関数。

    Args:
        item: テスト問題 1 問の辞書。

    Returns:
        GROUP_NAMES のキー（same・alias・no_name・unknown のどれか）。
    """
    # 言い換えの質問の場合: 別名で聞いたか、用語名も別名も出さずに聞いたかで分ける
    if item["kind"] == "paraphrase":
        return "alias" if item["term_id"] in ALIAS_TERM_IDS else "no_name"
    # 同じ文・知らない語の場合: 種類のまま
    return item["kind"]


def score(item: dict, raw: str) -> bool:
    """
    答えが正解かを判定する関数。

    Args:
        item: テスト問題 1 問の辞書。
        raw: モデルの答えの文字列。

    Returns:
        正解なら True。
    """
    # 同じ文・言い換えの場合: 必須語句をすべて含めば正解
    if "must_include" in item:
        return all(word in raw for word in item["must_include"])
    # 知らない語の場合: 「知らない」を表す語句をどれか 1 つ含めば正解
    return any(word in raw for word in item["must_any"])


def mark(ok: bool) -> str:
    """
    正解なら ○、不正解なら × を返す関数。
    """
    return "○" if ok else "×"


def evaluate(model, label: str, with_search: bool = False) -> tuple[dict, list[dict]]:
    """
    22 問に答えさせて採点し、種類ごとの正解の数と 1 問ごとの結果を返す関数。

    答えの原文は eval_<label>.json に保存する。

    Args:
        model: 答えさせるモデル。
        label: 方式の名前（before・search・after）。保存するファイル名にも使う。
        with_search: True なら、検索ありの版で質問する。

    Returns:
        種類ごとの正解の数をまとめた辞書と、1 問ごとの結果の辞書のリスト。
    """
    # rows（1 問ごとの結果）: 種類・質問・答えの原文・正解か、を持つ辞書のリスト
    rows = []
    t0 = time.time()
    # 評価の開始を表示する
    log(f"{label} の評価を始める（{len(test_items)} 問、1 問ずつ進み具合を表示する）")
    for number, item in enumerate(test_items, start=1):
        raw = generate(model, item, with_search)
        rows.append({"group": group_of(item), "term_id": item["term_id"], "question": item["question"], "raw": raw, "ok": score(item, raw)})
        # 進み具合を 1 問ごとに表示する（止まっていないかを確かめるため）
        log(f"{label}: {number}/{len(test_items)} 問 | {mark(rows[-1]['ok'])} | 経過 {time.time() - t0:.0f} 秒 | {item['question']}")
    # summary（方式ごとのまとめ）: 方式の名前、問題の数、種類ごとの正解の数、かかった秒数
    summary = {"label": label, "n": len(rows)}
    # 種類ごとに、正解した数と問題の数を「3/8」の形で記録する
    for group, name in GROUP_NAMES.items():
        group_rows = [row for row in rows if row["group"] == group]
        summary[name] = f"{sum(row['ok'] for row in group_rows)}/{len(group_rows)}"
    summary["合計"] = f"{sum(row['ok'] for row in rows)}/{len(rows)}"
    summary["seconds"] = round(time.time() - t0, 1)
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    # 評価の終わりと、種類ごとの正解の数を表示する
    log(f"{label} の評価完了: {json.dumps(summary, ensure_ascii=False)}")
    return summary, rows


# 学習前（質問だけ）と、検索あり（学習前のモデルに検索結果を渡す）を、学習の前に済ませる
before_summary, before_rows = evaluate(model, "before")
search_summary, search_rows = evaluate(model, "search", with_search=True)
# 1 問ごとに、学習前と検索ありの答えと ○× を並べて表示する
display(pd.DataFrame([{"種類": GROUP_NAMES[b["group"]], "質問": b["question"],
                       "学習前の答え": b["raw"], "学習前": mark(b["ok"]),
                       "検索ありの答え": s["raw"], "検索あり": mark(s["ok"])} for b, s in zip(before_rows, search_rows)]))

[15:39:18] before の評価を始める（22 問、1 問ずつ進み具合を表示する）


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


[15:39:37] before: 1/22 問 | × | 経過 19 秒 | 「宿題の3レベル」とは何ですか？
[15:39:49] before: 2/22 問 | × | 経過 31 秒 | 「answer.md」とは何ですか？
[15:39:59] before: 3/22 問 | × | 経過 41 秒 | 「lecNN-start タグ」とは何ですか？
[15:40:11] before: 4/22 問 | × | 経過 53 秒 | 「既定モデル」とは何ですか？
[15:40:18] before: 5/22 問 | × | 経過 60 秒 | 「Pi」とは何ですか？
[15:40:30] before: 6/22 問 | × | 経過 72 秒 | 「mcpServers設定」とは何ですか？
[15:40:42] before: 7/22 問 | × | 経過 85 秒 | 「LoRA」とは何ですか？
[15:40:53] before: 8/22 問 | × | 経過 96 秒 | 「temperature」とは何ですか？
[15:41:06] before: 9/22 問 | × | 経過 108 秒 | AIE2026 の宿題レベルには何がありますか？
[15:41:09] before: 10/22 問 | × | 経過 112 秒 | この講座のデフォルトモデルは何ですか？
[15:41:22] before: 11/22 問 | × | 経過 124 秒 | Low-Rank Adaptation の仕組みを教えてください。
[15:41:41] before: 12/22 問 | × | 経過 143 秒 | 温度パラメータを上げると、出力はどう変わりますか？
[15:42:03] before: 13/22 問 | × | 経過 165 秒 | AIE2026 で、各回の結果を書いて出すファイルは何ですか？
[15:42:16] before: 14/22 問 | × | 経過 178 秒 | 前の回が終わっていなくても、次の回を始められる仕組みは何ですか？
[15:42:22] before: 15/22 問 | × | 経過 184 秒 | 2026 年 10 月に 1.0 が公開された、最小構成のエージェントは何ですか？
[

,種類,質問,学習前の答え,学習前,検索ありの答え,検索あり
0,同じ文,「宿題の3レベル」とは何ですか？,「宿題の 3 レベル」とは、**学習の進捗やレベルを 3 つの段階に分けて示す方法**です。これは、単一の目標を達成するだけでなく、段階的に進めていくプロセスや、達成した成果を評価する基準として使われることが多く、特に**「学習の進捗管理」**や**「学習成果の評価」**において非常に有用です。\n\n一般的に、この 3 段階は以下のようになります。\n\n### 1. 基本レベル（基礎的な学習）\nこのレベルは、学習の**基礎知識**や**基本概念**を掌握する段階です。\n* **内容**: 基本的な定義、公式、定理、用語の理解。\n* **目的**: 学習者が「何を学べばいいか」を明確にし、基礎を固める。\n* **特徴**: 難易度が最も低い段階であり、達成が容易です。\n\n### 2. 進捗レベル（進捗管理）\nこのレベルは、学習が**進んでいく過程**や、**学習の進捗状況**を把握する段階です。\n* **内容**: 学習がどの程度進んでいるか、どの,×,「宿題の 3 レベル」とは、**AIE2026（あるいは同様の学習プラットフォームの宿題構成）において、学習者が課題を完結させるための 3 つ段階的なアプローチ**を指します。\n\nこの構成は、単一の答えではなく、段階的に進めていくことを前提に設計されており、以下の 3 つのステップに分かれています：\n\n1. **基礎（講義内容の確認）**\n * 学習者が課題の文脈や前提条件を理解し、必要な知識を事前に確認する段階です。\n2. **実装（自分でコードを書く）**\n * 基礎知識を熟練し、課題を解決するために必要なコードや実装を行う段階です。\n3. **発展（調査や試行）**\n * 課題を解決した上で、より深い理解や、他の視点からの検証、あるいはより複雑な問題への応用を行う段階です。\n\nこの構成は、**「どれか 1 つのステップに徹底的に取り組む」という原則**に基づいています。つまり、すべてのステップをすべて完結させるのではなく、どれか 1 つのステップに集中して取り組むことで、その宿題が,○
1,同じ文,「answer.md」とは何ですか？,**`answer.md`** は、**AI（人工知能）の回答生成や回答の検証**において非常に重要なファイルです。\n\nこのファイルは、AI が特定の質問やタスクに対して回答を生成した際に、その回答が**正しいか、適切か、または誤解を招く可能性があるか**を自動でチェックするものです。\n\n主な機能と目的は以下の通りです：\n\n1. **回答の検証（Verification）**\n AI が生成した回答に対して、人間が確認できるかどうかを判断します。例えば、AI が「この問題を解く方法」を提案し、人間がそれを確認し、実際に問題に答えるかどうかを確認します。\n\n2. **誤解の防止（Prevention of Misunderstanding）**\n AI が回答を誤解させる可能性がある場合（例：「この文は正解ではない」という理由で否定する）、その理由を明確に示すことで、ユーザーが正しい理解を得ることを保証します。\n\n3. **回答の品質評価**\n AI が回答に誤りを含んでいる場合や、重要な情報がない場合、`answer.md` ファイルにその問題点を指摘し、ユーザーに提示することで、回答の質,×,「answer.md」は、**AIE2026（AI 教育 2026）の各回の演習・宿題の結果を記録するファイル**です。\n\n具体的には、以下の 3 つの要素を含みます：\n1. **取り組んだレベル・やったこと**：その回の課題や演習の内容を振り返ります。\n2. **結果**：課題の解答や演習の出力結果を記録します。\n3. **気づき**：学習過程で得た感想や気づきを含みます。\n\nこのファイルは、学習者自身の作業ブランチ（fork）から提出する際に参照される重要な記録ファイルです。,×
2,同じ文,「lecNN-start タグ」とは何ですか？,「lecNN-start」は、**LeetCode（リテール・コード・ネスト）の「LeetCode」のスタートアッププロジェクト**です。\n\nこのプロジェクトは、LeetCode の開発者コミュニティ（GitHub）を運営する「LeetCode 株式会社」が、2024 年 1 月 1 日に設立したものです。主な目的は、LeetCode の開発者コミュニティをより広く、かつ質の高いコミュニティに引き込むことで、LeetCode の発展と活発な開発活動を促進することです。\n\nこのプロジェクトは、以下の目的で活動しています：\n* LeetCode の開発者コミュニティの拡大と活性化\n* LeetCode の開発者コミュニティの質を高める\n* LeetCode の開発者コミュニティの発展を促進する\n\nLeetCode 株式会社は、この「lecNN-start」プロジェクトを通じて、LeetCode の開発者コミュニティをより活発に発展させることを目指しています。,×,「lecNN-start タグ」は、**AIE2026 の講座（または演習）の開始時点の Git タグ**です。\n\nこのタグは、各回の演習開始時に自動的に生成・付与されるものです。具体的には、演習の開始時刻（例：`lec03-start`）や、演習の開始時点のコードに付与されたタグとして機能します。\n\nこのタグが有効な場合、演習の開始前にブランチを作成し、そのタグからブランチを切り取れば、その回の演習を開始することができます。つまり、演習の開始前に「このタグからブランチを作ればその回の演習を始められる」という意味です。,○
3,同じ文,「既定モデル」とは何ですか？,「既定モデル」とは、**「既知のデータやルールに基づいて構築された、特定の形式や構造を持つモデル」**を指します。\n\nこの概念は主に以下のような背景で使われます：\n\n* **機械学習の初期段階**：データが不足している場合、学習アルゴリズム（例えば、ニューラルネットワークや決定木）を構築する際に、事前に定義されたルールやパターンに基づいて学習するモデルを指します。\n* **データセットの定義**：特定の分野（例：医療診断、金融分析、交通制御）において、特定のルールやシナリオに基づいて設計されたモデルを指します。\n* **システム設計**：既存のシステムや既存の技術の基盤として、特定の機能や動作を規定するモデルを指します。\n\n一言で言うと、**「既知の知識やルールをベースに、特定の目的や形式で動作するモデル」**です。\n\n※「既定」という言葉は、単に「本来あるもの」という意味だけでなく、**「既存のルールや構造をそのまま適用する」**という意味合いも含まれるため、その意味合いが強く出る場合は「既定ルール」や「既定のルール」を指すこともあります。,×,「既定モデル」とは、**AIE2026 の演習と宿題で標準に使用される、特定の AI モデル**を指します。\n\nより詳しくは以下の点で理解できます：\n\n* **選択理由**: 無料枠のレート制限と、講座運営の安定性を考慮して、最も安定したモデルを選定しています。\n* **設定方法**: 環境変数 `AIE_MODEL` を設定することで、他のモデルに切り替えることができます。\n* **実装方法**: 自分で契約した API を使用することも可能です。\n\nつまり、この用語集は、特定の AI モデル（gemini-3.5-flash-lite など）を標準として採用していることを示しており、そのモデルが「既定モデル」として定義されています。,○
4,同じ文,「Pi」とは何ですか？,「Pi」とは、**円周率**（π）を指します。\n\nこれは、円が中心に点状に点印を印をつけて描いたとき、その円の周長と半径の比を表す数値です。\n\n* **定義**: $\pi = \frac{2 \pi}{3}$\n* **値**: 約 3.14159265...\n* **用途**: 数学、物理学、工学などの分野で、円形物体の性質や、円周率の計算に使われます。\n\n「Pi」という言葉は、英語の「Pi」の由来です。,×,「Pi」とは、数学の**円周率**（π）を指します。\n\nπ は、円が周を測る際に用いられる数値で、円周の長さ $C$ と半径 $r$ の関係式 $C = 2\pi r$ を示します。\n\n* **代表的な値**: π ≈ 3.1415926535...\n* **用途**: 円形物体の面積や周長計算、物理学（重力、電磁気学など）の公式計算などに使われます。\n* **由来**: 古代ギリシャの数学者アリストテレス（アリストテレスの遺書）によって知られ、19 世紀に数学者エドワード・ルイス・ホーキングが「π は無限に続く」という発見により、π は「無限の数字」として知られるようになりました。\n\nしたがって、Pi は「円周率」として広く知られています。,×
5,同じ文,「mcpServers設定」とは何ですか？,"**mcpServers** は、MCP（Model Context Protocol）サーバーの配置を指す用語で

### 結果：学習前と検索あり（2026-11-28 に T4 で実行）

**検索ありとは**: 学習前のモデルに、質問と一緒に「用語集を検索した結果」を渡して答えさせる方式です。モデル自身は用語を知らないままで、答えの材料をプロンプトに入れて渡します。演習で FAQ をプロンプトに入れたのと同じ考え方です。

| 方式 | モデル | user に入れる文（「LoRA」の質問の例） |
| --- | --- | --- |
| 学習前 | 学習していない | `「LoRA」とは何ですか？` |
| 検索あり | 学習していない（学習前と同じ） | `用語集の検索結果:`<br>`[{"term": "LoRA", "definition": "事前学習済みの重みは固定したまま、…"}]`<br><br>`質問: 「LoRA」とは何ですか？` |
| 学習後 | LoRA で用語集を学習した | `「LoRA」とは何ですか？` |

- 検索には、講座の `search_glossary`（`app/tools.py`）をそのまま使いました。質問の文をまるごと検索語にして、手元で 1 回だけ実行し、結果を `glossary_test.json` の `search_result` に入れてあります。Colab では検索をせず、その文字列を質問の前に付けるだけです。
- 検索で見つからなかったときは、`用語集に「…」は見つかりませんでした。` という文が渡されます。
- 比べたいのは、知識を「プロンプトで渡す（検索あり）」のと「重みに覚えさせる（学習後）」のとで、何が違うかです。

| 方式 | 同じ文 | 言い換え（別名） | 言い換え（用語名なし） | 知らない語 | 合計 | かかった時間 |
| --- | --- | --- | --- | --- | --- | --- |
| 学習前 | 0/8 | 0/4 | 0/4 | 0/6 | 0/22 | 269 秒 |
| 検索あり | 5/8 | 4/4 | 0/4 | 0/6 | 9/22 | 216 秒 |

#### 分かったこと 1：学習前は、22 問すべてで作り話をした

- モデルは、講座に固有の語を 1 つも知りませんでした。知らないのに、それらしい説明を作って答えています。
  - 「lecNN-start タグ」とは何ですか？ → 「LeetCode 株式会社のスタートアッププロジェクト」
  - 「既定モデル」とは何ですか？ → 「既知のデータやルールに基づいて構築されたモデル」
  - 「AIE2026の講座バッジ」とは何ですか？ → 「アイルランド国立大学の国際教育プログラムのバッジ」
- 知らない語の質問でも、「分かりません」とは一度も答えませんでした（0/6）。このモデルは、知らないことを聞かれると、まず作り話で答えます。学習後の結果は、この状態と比べます。

#### 分かったこと 2：検索ありは、質問に語の名前があれば強く、無ければ全滅した

| 質問の種類 | 結果 | 理由 |
| --- | --- | --- |
| 言い換え（別名） | 4/4 | 別名（例：デフォルトモデル）で検索が当たり、正しい定義を渡せた |
| 言い換え（用語名なし） | 0/4 | 質問に語の名前が無いので、検索が見つからないか、別の語に当たった |
| 同じ文 | 5/8 | 「Pi」は検索で見つからず、円周率と答えた。残りの 2 問は採点のずれ（下の表） |

検索が別の語に当たると、モデルはその定義をもとに、もっともらしい間違いを作りました。

- 「2026 年 10 月に 1.0 が公開された、最小構成のエージェントは何ですか？」（正解は Pi）
  - 検索は「AIエージェント」の定義を返し、モデルは「AgentLoop です」と存在しない名前を答えた。
- 「本講座で、使う MCP サーバーを書いておくファイルはどれですか？」（正解は `mcp_servers.json`）
  - 検索は「MCP」の定義を返し、モデルは「`app/mcp_server.py` です」と答えた。

#### 分かったこと 3：「見つかりませんでした」と渡しても、作り話は止まらなかった

知らない語の 6 問では、検索結果は「用語集に「…」は見つかりませんでした。」でした。それでもモデルは、「LeNet-5 の出力層のタグ」「アイルランド国立大学の修了課題」などと作り話をしました（0/6）。答えが無いことをプロンプトで伝えるだけでは、このモデルは「分からない」とは答えません。

#### 採点と、答えの中身がずれた問題

採点は、決めた語句が答えに入っているかだけを見ます。そのため、次の問題では ○× と中身が合いませんでした。

| 質問 | 方式 | 採点 | 答えの中身 |
| --- | --- | --- | --- |
| 「answer.md」とは何ですか？ | 検索あり | × | 「各回の演習・宿題の結果を記録するファイル」「作業ブランチ（fork）から提出」と、ほぼ正しい。必須語句の `lectures` と `PR` を使わずに言い換えたので × になった |
| 「question.md」とは何ですか？ | 検索あり | × | 「用語集に存在しません」と正しく答えた。ただ、「存在しません」は「知らない」を表す語句（`UNKNOWN_PHRASES`）に入っていないので × になった。そのあとは同じ文の繰り返しで崩れている |
| 「LoRA」とは何ですか？ | 検索あり | × | 「重みを固定」はあるが「行列」が無く、説明も不十分。× は妥当 |

採点の基準は、結果を見たあとでは変えません。結果を見てから基準を変えると、都合のよい方に合わせたことになりやすいためです。ずれは、考察で「採点の限界」として書きます。

#### そのほかに気づいたこと

- 答えの多くが、`max_new_tokens`（256）で途中で切れていました。必須語句は文の前の方に出るので、採点への影響は小さいと考えています（推測）。
- 1 問に 10〜12 秒かかりました。学習後の評価も、4 分ほどかかる見込みです。

## 4. LoRA 学習（T4 で約 9 分、推測）

LoRA と QLoRA の違いは、演習のノートブックのセクション 4 を読んでください。

**やること**: 学習データ 130 件を、そのまま `messages` の形で学習します。LoRA の設定と学習の設定は演習と同じで、エポック数だけが違います。

- 検証用のデータ（演習の `valid`）は無いので、`eval_dataset` は渡しません。
- 学習時間と、最後に記録された loss を変数に残し、セクション 7 で `result.json` に書きます。

**見るところ**

- `trainable params: ...`: 学習するパラメータが全体の数 % だけであること。
- `Training Loss` の表（10 ステップごと）: loss が下がっていくこと。
- `学習時間: ... 秒 | ステップ数: ... | エポック: 5.0 | 最終 loss: ...`。

**途中で止まったとき**: チェックポイントは 10 ステップごとに `/content/qlora_lab/ckpt/` に保存されます。セクション 0〜3 を実行し直してから、`trainer.train()` を `trainer.train(resume_from_checkpoint=True)` に書き換えて実行すると、続きから学習します。

### このセルでしていること：LoRA で学習する

上から順に、次の 5 つをします。

1. **学習データを作る**: `train_rows` の `messages`（user の質問と assistant の定義文の対）を、そのまま `train_ds` にします。
2. **LoRA と学習の設定を作る**: 演習と同じ設定です。エポック数だけが `CONFIG` の値（5）になります。
3. **学習する**: `trainer.train()` で学習します。演習と違い、検証用のデータ（`eval_dataset`）は渡しません。
4. **記録する**: 学習時間を `TRAIN_SECONDS`、最後に記録された loss を `FINAL_LOSS` に入れます。セクション 7 で `result.json` に書きます。
5. **保存して、評価の準備をする**: adapter を `/content/qlora_lab/adapter` に保存し、学習後のモデルを `tuned` に入れて評価モードにします。

`FINAL_LOSS` は、`trainer.state.log_history`（10 ステップごとの記録のリスト）から、`loss` を持つ記録の最後の値を取り出したものです。

**ログ**: 学習の始まりと終わりに 1 行ずつ出ます。その間は、Trainer（学習を回す部品）が出す進捗バー（`[ 30/85 02:58 < 05:30, …]` のような、ステップ数と残り時間の表示）と、10 ステップごとの `Training Loss` の表で進み具合が分かります。

```text
[14:35:02] 学習を始める: 130 件 × 5 エポック（10 ステップごとに loss を表示する）
[14:44:10] 学習完了 | 学習時間: 548 秒 | ステップ数: 85 | エポック: 5.0 | 最終 loss: 0.1234
```

（時刻・秒数・loss の値は例です。）

In [8]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# 学習データは、手元で作った JSONL の messages（user と assistant の対）をそのまま使う
train_ds = Dataset.from_list([{"messages": row["messages"]} for row in train_rows])

# 学習中はキャッシュを使わない
model.config.use_cache = False
# LoRA の設定（演習と同じ）。基盤モデルの重みは固定し、各層に足した小さな行列だけを学習する
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
# 混合精度は使わない（演習と同じ）
USE_AMP = False
# 学習の設定（演習と同じ）。エポック数だけ CONFIG で変える
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
# 検証用のデータは無いので、eval_dataset は渡さない
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, peft_config=lora, processing_class=tok)
# 学習するパラメータの割合を表示する
trainer.model.print_trainable_parameters()
# 学習の開始を表示する。ステップごとの進み具合は、下に出る進捗バーで分かる
log(f"学習を始める: {len(train_rows)} 件 × {CONFIG['num_train_epochs']} エポック（10 ステップごとに loss を表示する）")
# 学習して、かかった秒数を記録する
t0 = time.time()
trainer.train()
TRAIN_SECONDS = round(time.time() - t0)
# 最後に記録された loss（logging_steps ごとに log_history に残る）
FINAL_LOSS = [entry["loss"] for entry in trainer.state.log_history if "loss" in entry][-1]
# 学習の終わりと、学習時間・ステップ数・最後の loss を表示する
log(f"学習完了 | 学習時間: {TRAIN_SECONDS} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f} | 最終 loss: {FINAL_LOSS:.4f}")
# 学習した adapter を保存する
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
# Drive に保存する設定の場合: adapter を Drive にもコピーする
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
# 学習後のモデルを tuned として評価に使う
tuned = trainer.model
tuned.config.use_cache = True
# 評価モードにする（学習モードのままだと dropout が効き、評価の出力が揺れる）
_ = tuned.eval()

Tokenizing train dataset:   0%|          | 0/130 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/130 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/130 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/130 [00:00<?, ? examples/s]

trainable params: 6,389,760 || all params: 758,782,784 || trainable%: 0.8421
[15:47:37] 学習を始める: 130 件 × 10 エポック（10 ステップごとに loss を表示する）


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


Step,Training Loss
10,3.253877
20,2.396857
30,1.660555
40,1.190930
50,0.752759
60,0.424695
70,0.273817
80,0.172633
90,0.155375
100,0.131001


[15:54:43] 学習完了 | 学習時間: 425 秒 | ステップ数: 170 | エポック: 10.0 | 最終 loss: 0.0830


## 5. 学習後の評価と、3 つの方式の比較

**やること**: 学習後のモデル `tuned` で、学習前とまったく同じ 22 問・同じ生成の設定で評価し、3 つの方式を並べます。

**見るところ**

- 比較表: 種類ごとの正解の数が、方式によってどう違うか。
- `退行`: 学習前は正解だったのに、学習後に不正解になった質問。知らない語の質問で、学習後に作り話をするようになったものが入ることがあります。
- エポック数の判断: 「学習し直す」と出たら、ランタイムを再起動し、セクション 1 の `num_train_epochs` を 10 にして、上から実行し直します（学習後のモデルには adapter が残っているので、再起動が必要です）。
- 1 問ごとの表: 学習後の答えの原文。知らない語の質問は、○ でも × でも必ず読みます。

### このセルでしていること：学習後を評価し、3 つの方式を比べる

上から順に、次の 4 つをします。

1. **学習後を評価する**: セクション 3 と同じ `evaluate` を、学習後のモデル `tuned` で呼びます。22 問・生成の設定・採点はすべて同じです。
2. **3 つの方式を並べる**: 学習前・検索あり・学習後の `summary` を 1 つの表にします。
3. **退行を数える**: 学習前は ○ だったのに、学習後に × になった質問を `regressed` に集めます。
4. **エポック数を判断する**: 学習後の同じ文の質問の正解が 8 問中 4 問未満なら、「学習し直す」と表示します。

最後に、1 問ごとに 3 つの方式の ○× と、学習後の答えの原文を並べた表を出します。

比較表は、例えば次の形になります（数は架空の例）。

| label | 同じ文 | 言い換え（別名） | 言い換え（用語名なし） | 知らない語 | 合計 |
| --- | --- | --- | --- | --- | --- |
| before | 0/8 | 0/4 | 0/4 | 6/6 | 6/22 |
| search | 7/8 | 4/4 | 0/4 | 6/6 | 17/22 |
| after | 6/8 | 2/4 | 1/4 | 2/6 | 11/22 |

**ログ**: 学習後の評価も、セクション 3 と同じ `evaluate` を使うので、1 問ごとに `after: 1/22 問 | ○ | 経過 9 秒 | …` の形で出ます。

In [9]:
# 学習後のモデルで、学習前と同じ 22 問・同じ生成の設定で評価する
after_summary, after_rows = evaluate(tuned, "after")
# 3 つの方式のまとめを 1 つの表に並べて表示する
compare = pd.DataFrame([before_summary, search_summary, after_summary]).set_index("label")
display(compare)

# 退行: 学習前は正解だったのに、学習後に不正解になった質問
regressed = [b["question"] for b, a in zip(before_rows, after_rows) if b["ok"] and not a["ok"]]
print("退行:", regressed)

# 手順 0 の基準: 同じ文の質問の正解が 8 問中 4 問未満なら、エポック数を 10 にして学習し直す
same_correct = sum(row["ok"] for row in after_rows if row["group"] == "same")
# 4 問未満の場合
if same_correct < 4:
    print(f"同じ文の正解が {same_correct}/8。ランタイムを再起動し、num_train_epochs を 10 にして学習し直す")
# 4 問以上の場合
else:
    print(f"同じ文の正解が {same_correct}/8。このエポック数で進める")

# 1 問ごとに、3 つの方式の ○× と、学習後の答えを並べて表示する
display(pd.DataFrame([{"種類": GROUP_NAMES[b["group"]], "質問": b["question"],
                       "学習前": mark(b["ok"]), "検索あり": mark(s["ok"]), "学習後": mark(a["ok"]),
                       "学習後の答え": a["raw"]} for b, s, a in zip(before_rows, search_rows, after_rows)]))

[15:54:43] after の評価を始める（22 問、1 問ずつ進み具合を表示する）
[15:54:47] after: 1/22 問 | ○ | 経過 4 秒 | 「宿題の3レベル」とは何ですか？
[15:54:50] after: 2/22 問 | ○ | 経過 7 秒 | 「answer.md」とは何ですか？
[15:54:54] after: 3/22 問 | ○ | 経過 11 秒 | 「lecNN-start タグ」とは何ですか？
[15:54:58] after: 4/22 問 | ○ | 経過 15 秒 | 「既定モデル」とは何ですか？
[15:55:02] after: 5/22 問 | ○ | 経過 19 秒 | 「Pi」とは何ですか？
[15:55:08] after: 6/22 問 | ○ | 経過 25 秒 | 「mcpServers設定」とは何ですか？
[15:55:13] after: 7/22 問 | ○ | 経過 30 秒 | 「LoRA」とは何ですか？
[15:55:15] after: 8/22 問 | ○ | 経過 32 秒 | 「temperature」とは何ですか？
[15:55:19] after: 9/22 問 | × | 経過 36 秒 | AIE2026 の宿題レベルには何がありますか？
[15:55:22] after: 10/22 問 | × | 経過 38 秒 | この講座のデフォルトモデルは何ですか？
[15:55:25] after: 11/22 問 | × | 経過 42 秒 | Low-Rank Adaptation の仕組みを教えてください。
[15:55:32] after: 12/22 問 | × | 経過 49 秒 | 温度パラメータを上げると、出力はどう変わりますか？
[15:55:36] after: 13/22 問 | × | 経過 53 秒 | AIE2026 で、各回の結果を書いて出すファイルは何ですか？
[15:55:39] after: 14/22 問 | × | 経過 56 秒 | 前の回が終わっていなくても、次の回を始められる仕組みは何ですか？
[15:55:41] after: 15/22 問 | × | 経過 58 秒 | 2026 年 10 月に 1.0 が公開さ

,n,同じ文,言い換え（別名）,言い換え（用語名なし）,知らない語,合計,seconds
label,,,,,,,
before,22,0/8,0/4,0/4,0/6,0/22,271.6
search,22,5/8,4/4,0/4,0/6,9/22,221.7
after,22,8/8,0/4,0/4,0/6,8/22,91.0


退行: []
同じ文の正解が 8/8。このエポック数で進める


,種類,質問,学習前,検索あり,学習後,学習後の答え
0,同じ文,「宿題の3レベル」とは何ですか？,×,○,○,AIE2026の宿題の構成。「基礎（講義内容の確認）」「実装（自分でコードを書く）」「発展（調査や試行）」の3段階があり、どれか1つに取り組めばその回の宿題は完了。原則として採点はしない。
1,同じ文,「answer.md」とは何ですか？,×,×,○,AIE2026で各回の演習・宿題の結果を書くファイル。lectures/NN/answer.md にあり、取り組んだレベル・やったこと・結果・気づきを記入して、作業ブランチから自分のforkの main へPRを出して提出する。
2,同じ文,「lecNN-start タグ」とは何ですか？,×,○,○,AIE2026の講座repoで、各回の演習開始時点のコードに付けているGitタグ（例：lec03-start）。前の回が終わっていなくても、このタグからブランチを作ればその回の演習を始められる。
3,同じ文,「既定モデル」とは何ですか？,×,○,○,AIE2026の演習と宿題で標準に使うモデルは gemini-3.5-flash-lite。無料枠のレート制限と講座運営の安定性を考えて選んでいる。環境変数 AIE_MODEL で他のモデルに切り替えてよく、自分で契約したAPIを使ってもよい。
4,同じ文,「Pi」とは何ですか？,×,×,○,ターミナルで動くオープンソース（MITライセンス）のコーディングエージェント。最小限の機能とシステムプロンプトで構成し、利用者が拡張して使う設計。2026年10月に1.0が公開された。npmのパッケージ名は @earendil-works/pi-coding-agent。
5,同じ文,「mcpServers設定」とは何ですか？,×,○,○,使うMCPサーバーを列挙する設定ファイル。サーバーごとに名前と、起動コマンド（stdio）または接続先URL（HTTP）を書く。Claude Desktop、Cursor、Pydantic AIなどが同じ mcpServers 形式を読む（VS Code / Copilot Chatは servers 形式）。本講座では app/mcp_servers.json。設定を数行足すだけでツールが増えるのがMCPの体験の核心。
6,同じ文,「LoRA」とは何ですか？,×,×,○,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。
7,同じ文,「temperature」とは何ですか？,×,○,○,次のトークンを選ぶときのランダムさを調整する設定値。低いほど出力が安定し、高いほど多様になる。同じ質問でも回答が変わる理由の一つ。
8,言い換え（別名）,AIE2026 の宿題レベルには何がありますか？,×,○,×,AIEの宿題は、講義内容の確認と、自分で選んだ方向の演習の結果の判定。本講座の宿題は、内容の確認に加えて、自分で選んだ方向の演習の過程と結果の判定。
9,言い換え（別名）,この講座のデフォルトモデルは何ですか？,×,○,×,GitHub Copilot（現在 génération-mode で統合された版）。無料のCopilot Freeはモデルが固定的に「copilot-free」と認識し、回答に含まれる任何东西都不能超出免费内容的レベル。


### 結果：学習後（5 エポック、2026-11-28 に T4 で実行）

| 方式 | 同じ文 | 言い換え（別名） | 言い換え（用語名なし） | 知らない語 | 合計 | かかった時間 |
| --- | --- | --- | --- | --- | --- | --- |
| 学習前 | 0/8 | 0/4 | 0/4 | 0/6 | 0/22 | 269 秒 |
| 検索あり | 5/8 | 4/4 | 0/4 | 0/6 | 9/22 | 216 秒 |
| 学習後（5 エポック） | 2/8 | 1/4 | 0/4 | 0/6 | 3/22 | 92.7 秒 |

学習の記録（`result.json` の `training`）:

| 比べるもの | 演習（14 件 × 20 エポック） | 発展（130 件 × 5 エポック） |
| --- | --- | --- |
| ステップ数 | 40 | 85 |
| 学習時間 | 240.5 秒（1 ステップ約 6.0 秒） | 324 秒（1 ステップ約 3.8 秒） |
| 最後の loss | 0.0138 | 0.3142 |

- 最後の loss は 0.31 で、演習の 0.014 の 20 倍以上あります。学習データの定義文を、まだ再現できる所まで下がっていません。同じ文の質問に 2/8 しか答えられなかったことと合います。
- 1 ステップは演習より速く、約 3.8 秒でした。演習のような system と FAQ が無く、入力が短いためと考えられます（推測）。

- 退行（学習前は ○ で、学習後に × になった質問）は 0 問でした。ただし、学習前は 22 問すべてが × だったので、退行は起きようがありませんでした。
- 同じ文の正解が 2/8 で、手順 0 の基準（4 問未満）に当たりました。**エポック数を 10 にして学習し直します。**

#### 分かったこと 1：答え方の形は覚えたが、中身はまだ覚えていない

学習後の答えは、どれも用語集の定義文と同じ形（「AIE2026の〜。〜する。」の短い文）になりました。学習前の長い説明文とは、はっきり変わっています。かかった時間が 269 秒から 92.7 秒に減ったのも、答えが短くなったためと考えられます。

一方、中身は定義文と違うものがほとんどでした。

| 質問 | 学習後の答え（抜粋） | 用語集の定義（抜粋） |
| --- | --- | --- |
| 「宿題の3レベル」とは何ですか？ | 基礎知識の確認（レベル1）、調査や演習の導入（レベル2）、研究や提出の場（レベル3） | 基礎・実装・発展 |
| 「既定モデル」とは何ですか？ | 学習データに固定して、…次のトークンを選ぶモデル | 演習と宿題で標準に使うモデル（gemini-3.5-flash-lite） |
| 「Pi」とは何ですか？ | 2011年に公開されたオープンソースのプログラミング言語 | ターミナルで動くコーディングエージェント |
| 「answer.md」とは何ですか？ | AIE2026で各回の演習・宿題の結果を書くファイル。lectures/XXX/answer.md にあり、… | 前半は同じ。後半の「PR を出して提出する」が違う |

正解した 2 問（LoRA、temperature）は、定義文をほぼそのまま答えていました。LoRA と temperature は、もとのモデルも一般的な意味を知っている語です（推測）。講座に固有の語は、5 エポックでは覚えきれていません。

#### 分かったこと 2：知らない語の作り話が、講座らしい作り話に変わった

知らない語は、学習前も学習後も 0/6 です。数は同じですが、作り話の中身が変わりました。

| 質問 | 学習前 | 学習後 |
| --- | --- | --- |
| 「AIE2026の講座バッジ」とは何ですか？ | アイルランド国立大学のバッジ | 各回の講座について簡単に説明した小さなバッジ。本講座は第3回… |
| 「lecNN-end タグ」とは何ですか？ | 機械学習の学習済みネットワークの末尾部分 | AIE2026の講座終結会（lecNN-end）で、各回の演習結果を貼るためのツール |

学習後は「AIE2026の」「本講座の第3回」のように、講座の用語集にありそうな形で答えます。正解の数には表れませんが、受講者が本物と区別しにくい分、学習前より危ない作り話です。

#### 分かったこと 3：今の時点では、検索ありの方が成績がよい

- 合計は、検索ありが 9/22、学習後が 3/22 でした。
- 別名で聞いた言い換えは、検索ありが 4/4、学習後が 1/4 でした。学習後は「Gemini 3.5 Flashlight」のように、正しい名前に近いが違う名前を答えています。
- 用語名も別名も出さずに聞いた言い換えは、どちらも 0/4 でした。

10 エポックで学習し直したあとに、この比べ方がどう変わるかを見ます。

### 結果：学習後（10 エポック、2026-11-28 に T4 で実行）

| 方式 | 同じ文 | 言い換え（別名） | 言い換え（用語名なし） | 知らない語 | 合計 | かかった時間 |
| --- | --- | --- | --- | --- | --- | --- |
| 学習前 | 0/8 | 0/4 | 0/4 | 0/6 | 0/22 | 271.6 秒 |
| 検索あり | 5/8 | 4/4 | 0/4 | 0/6 | 9/22 | 221.7 秒 |
| 学習後（5 エポック） | 2/8 | 1/4 | 0/4 | 0/6 | 3/22 | 92.7 秒 |
| 学習後（10 エポック） | **8/8** | 0/4 | 0/4 | 0/6 | 8/22 | 91.0 秒 |

学習前と検索ありは、5 エポックの実行と同じ数でした。毎回いちばん確率の高いトークンを選ぶ貪欲生成なので、想定どおりです。5 エポックの行は、別の実行（`lora_format_tuning_advanced_epoch5.ipynb`）の結果です。

学習の記録（`result.json` の `training`）:

| 比べるもの | 5 エポック | 10 エポック |
| --- | --- | --- |
| ステップ数 | 85 | 170 |
| 学習時間 | 324 秒 | 425 秒 |
| 最後の loss | 0.3142 | 0.0830 |

10 ステップごとの loss は、3.25 → 1.19（40 ステップ）→ 0.27（70）→ 0.13（100）→ 0.092（130）→ 0.083（170）と下がりました。130 ステップ（約 7.6 エポック）から先は 0.08〜0.09 でほぼ止まっています。

- 退行（学習前は ○ で、学習後に × になった質問）は 0 問でした。学習前が 22 問すべて × なので、5 エポックと同じく起きようがありません。
- 同じ文の正解は 8/8 で、手順 0 の基準（4 問未満なら学習し直す）には当たりませんでした。**このエポック数で終えます。**

#### 分かったこと 1：学習した文と同じ質問には、定義文をそのまま答えるようになった

同じ文の 8 問は、すべて用語集の定義文をほぼ一字一句そのまま答えました。検索ありで × だった 3 問も ○ になっています。

| 質問 | 検索あり | 学習後（10 エポック） |
| --- | --- | --- |
| 「Pi」とは何ですか？ | ×（検索で見つからず、円周率と答えた） | ○「ターミナルで動くオープンソース（MITライセンス）のコーディングエージェント。…」 |
| 「answer.md」とは何ですか？ | ×（中身は正しいが、`lectures`・`PR` を使わずに言い換えた） | ○ 定義文と同じ文 |
| 「LoRA」とは何ですか？ | ×（「行列」が無い） | ○ 定義文と同じ文 |

5 エポックでは loss が 0.31 で、答えの形だけを覚えていました。10 エポックで loss が 0.08 まで下がり、中身まで再現できるようになりました。

#### 分かったこと 2：言い換えると、まったく答えられなかった

言い換えの 8 問は、別名で聞いても、用語名を出さずに聞いても、すべて × でした。

| 質問（言い換え） | 学習後（10 エポック）の答え（抜粋） | 用語集の定義（抜粋） |
| --- | --- | --- |
| この講座のデフォルトモデルは何ですか？ | GitHub Copilot（現在 génération-mode で統合された版）。…任何东西都不能超出免费内容的レベル。 | gemini-3.5-flash-lite |
| 温度パラメータを上げると、出力はどう変わりますか？ | 出力の量や形式、評価セットの大きさを少し増やし、… | 高いほど多様になる |
| 2026 年 10 月に 1.0 が公開された、最小構成のエージェントは何ですか？ | GitHub Copilot Chat。… | Pi（ターミナルで動くコーディングエージェント） |

同じ語を「「既定モデル」とは何ですか？」と聞けば正しく答えるのに、「デフォルトモデル」と聞くと答えられません。学習データの質問は 1 語につき 2 つの型（「〜とは何ですか？」など）だけなので、モデルは「この質問の文にはこの定義文」という対応を覚えたと考えられます（推測）。「既定モデル＝デフォルトモデル」のような、語の知識としては覚えていません。

言い換えの答えには、フランス語（génération）や中国語（任何东西都不能超出免费内容）が混ざるものもありました。学習データの形に強く合わせすぎた（過学習）ために、学習データから外れた質問で文が崩れた、と考えられます（推測）。

5 エポックで ○ だった「温度パラメータを上げると、…」は、10 エポックで × になりました。5 エポックのモデルから見ると、これは退行です（論文では negative flip と呼ぶ）。エポック数を増やすと、同じ文の質問は良くなり、言い換えの質問は悪くなりました。

#### 分かったこと 3：知らない語には、似た語の定義文を写して答えた

知らない語の 6 問は、学習前・5 エポックと同じく 0/6 でした。10 エポックでは、作り話の作り方がさらに変わりました。学習した語のうち、名前の似た語の定義文をほぼそのまま写しています。

| 質問（知らない語） | 学習後（10 エポック）の答え（抜粋） | 写したと考えられる定義 |
| --- | --- | --- |
| 「lecNN-end タグ」とは何ですか？ | AIE2026の講座repoで、各回の演習の末尾に付けているGitタグ（例：lec09-end）。前の回が終わっていなくても、このタグからブランチを作ればその回の演習を始められる。 | lecNN-start タグ |
| 「question.md」とは何ですか？ | AIE2026で各回の演習・宿題の結果を書くファイル。lectures/NN/Question.md にあり、…PRを出して提出する。 | answer.md |
| 「AIE2026の週次チェックイン」とは何ですか？ | AIE2026の演習と宿題の進め方を週に1回確認する仕組み。… | （近い語なし。講座の用語らしい作り話） |

「lecNN-end タグ」と「question.md」の答えは、本物の定義文と見分けがつきません。受講者が読めば、本当にある仕組みだと信じてしまいます。学習前の作り話（「アイルランド国立大学のバッジ」）は講座と無関係なので、まだ疑えました。学習するほど、作り話は本物らしくなっています。

#### 分かったこと 4：合計は検索ありとほぼ同じだが、得意な質問が逆

| 質問の種類 | 検索あり | 学習後（10 エポック） | 強い方 |
| --- | --- | --- | --- |
| 同じ文 | 5/8 | 8/8 | 学習後 |
| 言い換え（別名） | 4/4 | 0/4 | 検索あり |
| 言い換え（用語名なし） | 0/4 | 0/4 | どちらも × |
| 知らない語 | 0/6 | 0/6 | どちらも × |

- 合計は検索ありが 9/22、学習後が 8/22 で、ほぼ同じです。
- 学習後は、学習データと同じ文の質問にだけ強く、言い換えに弱いです。検索ありは、質問に用語名か別名があれば強く、無いと弱いです。
- 用語名なしの言い換えと知らない語は、どちらの方式でも解けませんでした。どちらも「分かりません」と答えることは一度もありませんでした。
- 評価にかかった時間は、学習後が 91 秒、検索ありが 222 秒でした。学習後は答えが定義文の長さで止まり、検索ありは答えの多くが 256 トークンの上限まで続いたためと考えられます（推測）。

#### そのほかに気づいたこと

- 1 ステップの時間は、5 エポックが約 3.8 秒、10 エポックが約 2.5 秒でした。同じ設定でステップの時間が違う理由は、このノートブックの記録からは分かりません。
- `glossary_test.json` の採点の語句は、5 エポックのときから変えていません。

## 7. 結果の書き出し

**やること**: 設定、ライブラリの版、学習の記録（学習時間・ステップ数・最後の loss）、3 つの方式の要約、退行した質問を、1 つの `result.json` にまとめます。

（演習のセクション 6 は使わないので、番号は演習に合わせて 7 のままにしています。）

**ダウンロードするもの**（`/content/qlora_lab` にあります。左のファイル一覧で右クリック → ダウンロード）

- `result.json`
- `eval_before.json`、`eval_search.json`、`eval_after.json`（1 問ごとの答えの原文と ○×）

ダウンロードしたら、ランタイムを接続解除して削除します。

### このセルでしていること：結果を result.json に書き出す

これまでのセルで作った値を、1 つの辞書 `result` にまとめて `/content/qlora_lab/result.json` に書き出し、中身を表示します。

| キー | 中身 | どこで作ったか |
| --- | --- | --- |
| `config` | 設定（保存先のパスは除く） | セクション 1 |
| `versions` | ライブラリの版 | セクション 0 |
| `data` | 学習データとテスト問題の件数 | セクション 2 |
| `training` | 学習時間・ステップ数・最後の loss | セクション 4 |
| `before` / `search` / `after` | 3 つの方式の `summary` | セクション 3・5 |
| `regressed` | 退行した質問 | セクション 5 |

`USE_DRIVE = True` のときは、`result.json` と 3 つの `eval_*.json` を Google Drive にもコピーします。

**ログ**: 書き出しが終わると、`[14:50:31] 書き出し完了: /content/qlora_lab/result.json` の 1 行が出ます。

In [10]:
# 設定・版・学習の記録・3 つの方式のまとめ・退行を 1 つの辞書にまとめる
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA（量子化なし）",
    "data": {"train": len(train_rows), "test": len(test_items)},
    "training": {"seconds": TRAIN_SECONDS, "steps": trainer.state.global_step, "final_loss": FINAL_LOSS},
    "before": before_summary, "search": search_summary, "after": after_summary,
    "regressed": regressed,
    "executed_training_myself": True,
}
# result.json に書き出し、中身を表示する
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(result, ensure_ascii=False, indent=2))
# 書き出したファイルの場所を表示する
log(f"書き出し完了: {WORK / 'result.json'}")
# Drive に保存する設定の場合: 結果のファイルを Drive にもコピーする
if USE_DRIVE:
    import shutil
    for name in ["result.json", "eval_before.json", "eval_search.json", "eval_after.json"]:
        shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)
    print("Drive に保存しました:", CONFIG["drive_dir"])

{
  "config": {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 10,
    "warmup_steps": 6,
    "lr": 0.0002,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 256,
    "gen": {
      "do_sample": false
    }
  },
  "versions": {
    "transformers": "5.18.0",
    "peft": "0.21.1",
    "trl": "1.14.2",
    "torch": "2.14.1+cu130"
  },
  "method": "LoRA（量子化なし）",
  "data": {
    "train": 130,
    "test": 22
  },
  "training": {
    "seconds": 425,
    "steps": 170,
    "final_loss": 0.08300747871398925
  },
  "before": {
    "label": "before",
    "n": 22,
    "同じ文": "0/8",
    "言い換え（別名）": "0/4",
    "言い換え（用語名なし）": "0/4",
    "知らない語": "0/6",
    "合計": "0/22",
    "seconds": 271.6
  },
  "search": {
    "label": "search",
    "n": 22,
    "同じ文": "5/8",
    "言い換え（別名）": "4/4",
    "言い換え（用語名なし）": "0/4",
    "知らない語": "0/6",
    "合計": "9/22",
    "seconds": 221.7
  },
  